In [ ]:
!pip install h5py neurokit2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk
import h5py

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Identify and list folder in drive where Open Data Set is
!ls "drive/MyDrive/BMEN600/BMEN600-Group14/Dataset/SPH_DATA"

 code.csv						  metadata.csv
 Independent_AFib_Statistical_Report.xlsx		  records
'Independent_AFib_Statistical_Report.xlsx - Sheet1.csv'


In [ ]:
import h5py
import os
import shutil

drive_path = 'drive/MyDrive/BMEN600/BMEN600-Group14/Dataset/SPH_DATA/records/A00001.h5' #Path is dependent on where file is in your individual drive
local_path = '/content/local_records/A00001.h5'

# Dynamically copy the file to local fast storage if it doesn't exist yet
if not os.path.exists(local_path):
    if os.path.exists(drive_path):
        print("Local cache not found. Copying A00001.h5 from Google Drive to local storage...")
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        shutil.copy(drive_path, local_path)
        print("Copy complete.")
    else:
        print(f"Error: Source file not found in Google Drive path: {drive_path}")

# Open and inspect the local file
if os.path.exists(local_path):
    with h5py.File(local_path, 'r') as f:
        print("\nKeys available in this file:", list(f.keys()))
        for key in f.keys():
            if isinstance(f[key], h5py.Dataset):
                print(f"Dataset: '{key}' | Shape: {f[key].shape} | Datatype: {f[key].dtype}")


Keys available in this file: ['ecg']
Dataset: 'ecg' | Shape: (12, 5000) | Datatype: float16


In [ ]:
# ==============================================================================
# STEP 1: DEFINE PERMANENT GOOGLE DRIVE DIRECTORY PATHS
# ==============================================================================
# (This script assumes your drive is already mounted via drive.mount("/content/drive"))
base_dir = 'drive/MyDrive/BMEN600/BMEN600-Group14/Dataset/SPH_DATA/'
drive_records_dir = os.path.join(base_dir, 'records')
metadata_path = os.path.join(base_dir, 'metadata.csv')

In [ ]:
# ==============================================================================
# STEP 2: HIGH-SPEED TRANSFERS TO EPHEMERAL STORAGE (THE OPTIMIZATION FIX)
# ==============================================================================
local_records_dir = '/content/local_records'
os.makedirs(local_records_dir, exist_ok=True)

print("Transferring HDF5 database files to Colab local flash drive...")
# The -n flag ensures it only copies missing files if you run this cell again
# Fixed non-portable warning syntax: copies safely without overwriting
!cp -r --update=none "$drive_records_dir"/* "$local_records_dir/"

# Swap the active loop target pointer to point directly to the fast flash workspace
records_dir = local_records_dir
print(f"Transfer complete! Local files verified: {len(os.listdir(records_dir))}\n")

Transferring HDF5 database files to Colab local flash drive...
Transfer complete! Local files verified: 25770



In [ ]:
# ==============================================================================
# STEP 3: LOAD METADATA AND PREPROCESSING
# ==============================================================================
import numpy as np
from scipy.signal import butter, filtfilt
from tqdm.notebook import tqdm

metadata_df = pd.read_csv(metadata_path, index_col='ECG_ID')
print(f"Loaded master tracking log with {len(metadata_df)} available records.")

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    """
    Applies a zero-phase Butterworth bandpass filter to a 1D ECG lead.
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist

    if high >= 1.0:
        high = 0.99

    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def preprocess_12_lead_ecg(ecg_data, is_child=False, fs=500.0):
    """
    Applies bandwidth filtering and Z-score normalization to all 12 leads.
    """
    if ecg_data.shape[0] != 12:
        # Fallback if matrix is transposed as (samples, 12)
        if ecg_data.shape[1] == 12:
            ecg_data = ecg_data.T
        else:
            raise ValueError(f"ECG array must contain 12 rows. Shape found: {ecg_data.shape}")

    preprocessed_ecg = np.zeros_like(ecg_data, dtype=np.float64)
    lowcut = 0.05
    highcut = 250.0 if is_child else 150.0

    for lead_idx in range(12):
        lead_signal = ecg_data[lead_idx, :]

        # 1. Bandpass Filtering
        filtered_signal = butter_bandpass_filter(lead_signal, lowcut, highcut, fs)

        # 2. Z-Score Normalization
        mean_val = np.mean(filtered_signal)
        std_val = np.std(filtered_signal)

        if std_val == 0:
            preprocessed_ecg[lead_idx, :] = filtered_signal - mean_val
        else:
            preprocessed_ecg[lead_idx, :] = (filtered_signal - mean_val) / std_val

    return preprocessed_ecg

Loaded master tracking log with 25770 available records.


In [ ]:
# ==============================================================================
# STEP 4: DEFINE THE DETERMINISTIC CLASSIFICATION MODEL (12-LEAD UPGRADE)
# ==============================================================================
import neurokit2 as nk
from tqdm import tqdm

def evaluate_afib_statistically(lead_results):
    """
    Applies classical deterministic physiological thresholds to a multi-lead portfolio.
    Uses a majority-voting consensus approach to eliminate localized rhythm artifacts.
    """
    if not lead_results or len(lead_results) == 0:
        return "Insufficient Data/Artifact"

    votes = {"AFib": 0, "Non-AFib Irregular": 0, "Regular": 0}
    insufficient_count = 0

    # Evaluate every single available lead channel independently
    for res in lead_results:
        sample_count = res['Lead_Beats_Count']
        hr = res['Lead_HR']
        cv_rr = res['Lead_CV_RR']
        rmssd = res['Lead_RMSSD_ms']

        if sample_count < 10 or hr == 0 or np.isnan(hr):
            insufficient_count += 1
            continue

        # CLASSICAL AFIB PARAMETERS: Global irregularity (CV_RR >= 15%)
        # combined with high successive micro-volatility (RMSSD >= 65ms)
        if cv_rr >= 0.15 and rmssd >= 65.0:
            votes["AFib"] += 1

        # High macro variance but regular local tracking (e.g. Sinus Arrhythmia, PACs)
        elif cv_rr >= 0.08 and rmssd < 65.0:
            votes["Non-AFib Irregular"] += 1

        else:
            votes["Regular"] += 1

    # Total valid channels that provided clean information
    valid_leads = len(lead_results) - insufficient_count
    if valid_leads == 0:
        return "Insufficient Data/Artifact"

    # MAJORITY CONSENSUS LOGIC: Flag AFib only if it systematically dominates across vectors
    # This prevents isolated ectopic beats/PACs/PVCs from tripping a false positive
    if votes["AFib"] >= (valid_leads / 2):
        return "Atrial Fibrillation (Confirmed)"
    elif votes["Non-AFib Irregular"] > votes["Regular"]:
        return "Non-AFib Irregular Rhythm"
    else:
        return "Regular Sinus Rhythm"

def process_record_for_afib(file_path, is_child=False, sampling_rate=500):
    """
    Extracts time-domain variability statistics across the R-R interval profile
    for all 12 leads utilizing your new preprocessing and filtering logic.
    """
    import h5py
    try:
        with h5py.File(file_path, 'r') as f:
            ecg_matrix = np.array(f['ecg'])

        # 1. Run your updated custom STEP 3 Preprocessing pipeline (0.05Hz - 150/250Hz + Z-Score)
        cleaned_12_lead = preprocess_12_lead_ecg(ecg_matrix, is_child=is_child, fs=sampling_rate)

        lead_metrics_summary = []

        # 2. Iterate through all 12 leads using the filtered array
        for lead_idx in range(12):
            signal = cleaned_12_lead[lead_idx, :]

            # Delineate peak boundaries on the filtered signal
            cleaned = nk.ecg_clean(signal, sampling_rate=sampling_rate, method='neurokit')
            _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=sampling_rate, method='neurokit')

            peaks = rpeaks['ECG_R_Peaks']
            num_beats = len(peaks)

            if num_beats >= 4:
                # Generate R-R interval sequence mapped in milliseconds
                rr_intervals = np.diff(peaks) / sampling_rate * 1000

                # Interpolate heart rate over time window
                rate_signal = nk.ecg_rate(rpeaks, sampling_rate=sampling_rate, desired_length=len(cleaned))
                mean_hr = np.mean(rate_signal)

                mean_rr = np.mean(rr_intervals)
                sdnn = np.std(rr_intervals)
                cv_rr = sdnn / mean_rr if mean_rr > 0 else 0
                rmssd = np.sqrt(np.mean(np.square(np.diff(rr_intervals))))

                lead_metrics_summary.append({
                    'Lead_Index': lead_idx,
                    'Lead_HR': round(mean_hr, 1),
                    'Lead_CV_RR': round(cv_rr, 4),
                    'Lead_RMSSD_ms': round(rmssd, 2),
                    'Lead_Beats_Count': num_beats
                })
            else:
                # Append basic fallback metrics for flat/unreadable leads
                lead_metrics_summary.append({
                    'Lead_Index': lead_idx,
                    'Lead_HR': 0,
                    'Lead_CV_RR': 0,
                    'Lead_RMSSD_ms': 0,
                    'Lead_Beats_Count': num_beats
                })

        return lead_metrics_summary

    except Exception as e:
        pass
    return None


In [ ]:
# ==============================================================================
# STEP 4: ACTIVE BATCH ANALYSIS LOOP (12-LEAD INTEGRATED)
# ==============================================================================
import os
from tqdm.notebook import tqdm

afib_findings = []

print("Initializing high-speed independent 12-lead AFib statistical screening engine...")

# Ensure your metadata dataframe contains an 'Age' column

for ecg_id in tqdm(metadata_df.index, desc="Processing ECGs"):
    file_name = f"{ecg_id}.h5"
    file_path = os.path.join(records_dir, file_name)

    if os.path.exists(file_path):
        # 1. Dynamically determine if the patient is a child from the metadata log
        try:
            patient_age = metadata_df.loc[ecg_id, 'Age']
            is_child = True if patient_age < 18 else False
        except Exception:
            is_child = False  # Safe baseline fallback if age is missing or NaN

        # 2. Extract metrics across all 12 leads (drops old lead_idx parameter)
        lead_metrics_summary = process_record_for_afib(file_path, is_child=is_child, sampling_rate=500)

        if lead_metrics_summary:
            # 3. Pass the full 12-lead portfolio to the new majority voting evaluator
            classification = evaluate_afib_statistically(lead_metrics_summary)

            # 4. Package the results cleanly for your final tracking log
            record_summary = {
                'ECG_ID': ecg_id,
                'Statistical_Classification': classification,
                'Is_Child_Filtered': is_child
            }

            # Flatten Lead II and Lead V1 features into the row for easy downstream viewing
            # (Ensures your final table still contains the heart rate and volatility values)
            for res in lead_metrics_summary:
                if res['Lead_Index'] in [1, 6]: # Lead II (1) and Lead V1 (6)
                    suffix = 'Lead_II' if res['Lead_Index'] == 1 else 'Lead_V1'
                    record_summary[f'HR_{suffix}'] = res['Lead_HR']
                    record_summary[f'CV_RR_{suffix}'] = res['Lead_CV_RR']
                    record_summary[f'RMSSD_{suffix}'] = res['Lead_RMSSD_ms']

            afib_findings.append(record_summary)

print(f"Batch processing complete! Successfully analyzed {len(afib_findings)} files.")

Initializing high-speed independent 12-lead AFib statistical screening engine...


Processing ECGs:   0%|          | 0/25770 [00:00<?, ?it/s]

Batch processing complete! Successfully analyzed 25768 files.


In [ ]:
# ==============================================================================
# STEP 5: MERGE DATASETS AND EXPORT REPORT TO DRIVE
# ==============================================================================
import pandas as pd
import os

# Convert findings into a tracking DataFrame
findings_df = pd.DataFrame(afib_findings)

if not findings_df.empty:
    findings_df.set_index('ECG_ID', inplace=True)

    # Using 'left' join preserves all original metadata rows; missing files will gracefully show as NaN
    final_report_df = metadata_df.join(findings_df, how='left')

    # Fill unanalyzed/missing files explicitly so they don't break summary value counts
    final_report_df['Statistical_Classification'] = final_report_df['Statistical_Classification'].fillna('File Missing/Not Analyzed')

    # Write final analytical spreadsheet report right back to your permanent Drive base path
    output_path = os.path.join(base_dir, 'Independent_AFib_Statistical_Report.xlsx')
    final_report_df.to_excel(output_path)

    print(f"\nProcessing successful! Statistical Excel report saved to permanent storage path:\n--> {output_path}")
    print("\nObserved Screening Results Distribution Summary:")
    print(final_report_df['Statistical_Classification'].value_counts())

    # Quick sanity check print to track your false positive optimization progress
    confirmed_afib = (final_report_df['Statistical_Classification'] == "Atrial Fibrillation (Confirmed)").sum()
    print(f"\nTotal cases flagged as AFib by this traditional 12-lead implementation: {confirmed_afib} (Target: ~675)")

else:
    print("\n[ERROR]: The afib_findings list is entirely empty. Please check that your file paths in Step 4 are correct.")



Processing successful! Statistical Excel report saved to permanent storage path:
--> drive/MyDrive/BMEN600/BMEN600-Group14/Dataset/SPH_DATA/Independent_AFib_Statistical_Report.xlsx

Observed Screening Results Distribution Summary:
Statistical_Classification
Regular Sinus Rhythm               22315
Atrial Fibrillation (Confirmed)     2551
Insufficient Data/Artifact           732
Non-AFib Irregular Rhythm            170
File Missing/Not Analyzed              2
Name: count, dtype: int64

Total cases flagged as AFib by this traditional 12-lead implementation: 2551 (Target: ~675)
